# Protein ↔ Pathway Relation-Wise Merge

Merges Protein–Pathway triples from CrossBAR (×2) and TARKG; fills protein head names
from UniProt; deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [1]:
import os
import pandas as pd
import numpy as np

BASE_DIR  = '/storage/Arushi/090526_EvoAge/kg_formation/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'
PROC_DIR  = BASE_DIR + 'processed_data/'

OUT_PATH  = BASE_DIR + 'processed_data_relation_wise_merge/generalised/PROTEIN_PATHWAY/ALL_PROTEIN_PATHWAY.parquet'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Protein Lookup Dictionaries — UniProt

In [2]:
# RecName dict (for sources that store UniProt ACs in head)
Uniprot_Recname = pd.read_csv(DB_DIR + 'uniprot/Uniprot_ID_Recname.csv')
Uniprot_Recname_dict = dict(zip(Uniprot_Recname['AC'], Uniprot_Recname['RecName']))

# Parsed TrEMBL dict (AC → NAME) for head-name fallback
uniprot = pd.read_csv(DB_DIR + 'uniprot/uniprot_parsed_results.csv')
uniprot_trembl_AC_Name_dict = dict(zip(uniprot['AC'], uniprot['NAME']))
del uniprot

print(f"UniProt RecName dict: {len(Uniprot_Recname_dict):,} | TrEMBL dict: {len(uniprot_trembl_AC_Name_dict):,}")

UniProt RecName dict: 794,151 | TrEMBL dict: 252,635,149


## 2. Load KG Sources

### 2a. CrossBAR (×2)

The second file maps head ACs to RecNames at load time.

In [3]:
CrossBAR_Protein_Pathway = pd.read_csv(PROC_DIR + 'crossbar/Protein_Pathway_Crossbar.csv')
CrossBAR_Protein_Pathway.columns = CrossBAR_Protein_Pathway.columns.str.lower()

CrossBAR_Protein_Pathway2 = pd.read_csv(PROC_DIR + 'crossbar/Protein_Pathway_Crossbar_2.csv')
CrossBAR_Protein_Pathway2.columns = CrossBAR_Protein_Pathway2.columns.str.lower()
CrossBAR_Protein_Pathway2['head_detail_name'] = CrossBAR_Protein_Pathway2['head'].map(Uniprot_Recname_dict)

print(f"CrossBAR 1: {len(CrossBAR_Protein_Pathway):,} rows")
print(f"CrossBAR 2: {len(CrossBAR_Protein_Pathway2):,} rows")

CrossBAR 1: 36,496 rows
CrossBAR 2: 19,474 rows


In [4]:

CrossBAR_Protein_Pathway['kg_type'] = 'Generalised'
CrossBAR_Protein_Pathway['kg_source'] = 'crossbar'
CrossBAR_Protein_Pathway['species'] = 'Homo species'
CrossBAR_Protein_Pathway

,head,relation,tail,head_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,Q8NGE3,Protein_Pathway,R-HSA-381753,Protein,Pathway,crossbar,Olfactory receptor 10P1,Olfactory Signaling Pathway,Uniprot_ID,Reactome_ID,Generalised,Homo species
1,Q9NWB1,Protein_Pathway,R-HSA-9022707,Protein,Pathway,crossbar,RNA binding protein fox-1 homolog 1,MECP2 regulates transcription factors,Uniprot_ID,Reactome_ID,Generalised,Homo species
2,Q13315,Protein_Pathway,R-HSA-5693548,Protein,Pathway,crossbar,Serine-protein kinase ATM,Sensing of DNA Double Strand Breaks,Uniprot_ID,Reactome_ID,Generalised,Homo species
3,Q13315,Protein_Pathway,R-HSA-69541,Protein,Pathway,crossbar,Serine-protein kinase ATM,Stabilization of p53,Uniprot_ID,Reactome_ID,Generalised,Homo species
4,Q13315,Protein_Pathway,R-HSA-9664873,Protein,Pathway,crossbar,Serine-protein kinase ATM,Pexophagy,Uniprot_ID,Reactome_ID,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...
36491,Q92539,Protein_Pathway,R-HSA-75109,Protein,Pathway,crossbar,Phosphatidate phosphatase LPIN2 {ECO:0000305},Triglyceride biosynthesis,Uniprot_ID,Reactome_ID,Generalised,Homo species
36492,Q92539,Protein_Pathway,R-HSA-4419969,Protein,Pathway,crossbar,Phosphatidate phosphatase LPIN2 {ECO:0000305},Depolymerisation of the Nuclear Lamina,Uniprot_ID,Reactome_ID,Generalised,Homo species
36493,Q92539,Protein_Pathway,R-HSA-1483191,Protein,Pathway,crossbar,Phosphatidate phosphatase LPIN2 {ECO:0000305},Synthesis of PC,Uniprot_ID,Reactome_ID,Generalised,Homo species
36494,P62851,Protein_Pathway,R-HSA-72695,Protein,Pathway,crossbar,Small ribosomal subunit protein eS25 {ECO:0000...,"Formation of the ternary complex, and subseque...",Uniprot_ID,Reactome_ID,Generalised,Homo species


In [5]:
CrossBAR_Protein_Pathway2['kg_type'] = 'Generalised'
CrossBAR_Protein_Pathway2['kg_source'] = 'crossbar'
CrossBAR_Protein_Pathway2['species'] = 'Homo species'
CrossBAR_Protein_Pathway2

,head,relation,tail,head_type,tail_type,kg_source,head_detail_name,tail_alt_id,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,Q8NGE3,Protein_Pathway,hsa04740,Protein,Pathway,crossbar,Olfactory receptor 10P1,hsa04740,Olfactory transduction,Uniprot_ID,KEGG_ID,Generalised,Homo species
1,Q13315,Protein_Pathway,hsa03440,Protein,Pathway,crossbar,Serine-protein kinase ATM,hsa03440,Homologous recombination,Uniprot_ID,KEGG_ID,Generalised,Homo species
2,Q13315,Protein_Pathway,hsa04115,Protein,Pathway,crossbar,Serine-protein kinase ATM,hsa04115,p53 signaling pathway,Uniprot_ID,KEGG_ID,Generalised,Homo species
3,Q13315,Protein_Pathway,hsa04064,Protein,Pathway,crossbar,Serine-protein kinase ATM,hsa04064,NF-kappa B signaling pathway,Uniprot_ID,KEGG_ID,Generalised,Homo species
4,Q13315,Protein_Pathway,hsa04110,Protein,Pathway,crossbar,Serine-protein kinase ATM,hsa04110,Cell cycle,Uniprot_ID,KEGG_ID,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
19469,Q92539,Protein_Pathway,hsa00564,Protein,Pathway,crossbar,Phosphatidate phosphatase LPIN2 {ECO:0000305},hsa00564,Glycerophospholipid metabolism,Uniprot_ID,KEGG_ID,Generalised,Homo species
19470,Q92539,Protein_Pathway,hsa04150,Protein,Pathway,crossbar,Phosphatidate phosphatase LPIN2 {ECO:0000305},hsa04150,mTOR signaling pathway,Uniprot_ID,KEGG_ID,Generalised,Homo species
19471,Q8WWI1,Protein_Pathway,hsa04520,Protein,Pathway,crossbar,LIM domain only protein 7,hsa04520,Adherens junction,Uniprot_ID,KEGG_ID,Generalised,Homo species
19472,P62851,Protein_Pathway,hsa03010,Protein,Pathway,crossbar,Small ribosomal subunit protein eS25 {ECO:0000...,hsa03010,Ribosome,Uniprot_ID,KEGG_ID,Generalised,Homo species


### 2b. TARKG

`head_id_is` / `tail_id_is` are swapped in the raw file — corrected here.

In [6]:
TARKG_Protein_Pathway = pd.read_csv(PROC_DIR + 'TARKG/Protein_Pathway_TARKG.csv')
TARKG_Protein_Pathway.columns = TARKG_Protein_Pathway.columns.str.lower()

print(f"TARKG:      {len(TARKG_Protein_Pathway):,} rows")

TARKG_Protein_Pathway['kg_type'] = 'Generalised'
TARKG_Protein_Pathway['kg_source'] = 'TARKG'
TARKG_Protein_Pathway['species'] = 'Homo species'

TARKG_Protein_Pathway.head(2)

TARKG:      38,882 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_index,kg,change,kg_type,species
0,Q8WYP5,Protein_Pathway,R-HSA-2467813,Protein,PROTEIN_PATHWAY_ASSOCIATION,Pathway,TARKG,Protein ELYS,Separation of Sister Chromatids,Uniprot_ID,Reactome_ID,BioKG-302944,BioKG,0,Generalised,Homo species
1,Q8NFH4,Protein_Pathway,R-HSA-2467813,Protein,PROTEIN_PATHWAY_ASSOCIATION,Pathway,TARKG,Nucleoporin Nup37,Separation of Sister Chromatids,Uniprot_ID,Reactome_ID,BioKG-110254,BioKG,0,Generalised,Homo species


## 3. Check and Fix Duplicate Columns

In [7]:
SOURCE_DFS = [
    ('CrossBAR_Protein_Pathway',  CrossBAR_Protein_Pathway),
    ('CrossBAR_Protein_Pathway2', CrossBAR_Protein_Pathway2),
    ('TARKG_Protein_Pathway',     TARKG_Protein_Pathway),
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[CrossBAR_Protein_Pathway] ✓ no duplicates
[CrossBAR_Protein_Pathway2] ✓ no duplicates
[TARKG_Protein_Pathway] ✓ no duplicates


In [8]:
CrossBAR_Protein_Pathway  = CrossBAR_Protein_Pathway.loc[:,  ~CrossBAR_Protein_Pathway.columns.duplicated(keep='first')]
CrossBAR_Protein_Pathway2 = CrossBAR_Protein_Pathway2.loc[:, ~CrossBAR_Protein_Pathway2.columns.duplicated(keep='first')]
TARKG_Protein_Pathway     = TARKG_Protein_Pathway.loc[:,     ~TARKG_Protein_Pathway.columns.duplicated(keep='first')]

for name, df in SOURCE_DFS:
    remaining = df.columns[df.columns.duplicated()].tolist()
    print(f"[{name}] {'✓ clean' if not remaining else '✗ dupes: ' + str(remaining)}")

[CrossBAR_Protein_Pathway] ✓ clean
[CrossBAR_Protein_Pathway2] ✓ clean
[TARKG_Protein_Pathway] ✓ clean


## 4. Align Schemas and Concatenate

In [9]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name','kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    tmp['head_id_is'] = tmp['head_id_is'].astype(str)
    tmp['tail_id_is'] = tmp['tail_id_is'].astype(str)
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 94,852 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,Q8NGE3,Protein_Pathway,R-HSA-381753,Protein,NaN,Pathway,crossbar,Uniprot_ID,Reactome_ID,Olfactory receptor 10P1,Olfactory Signaling Pathway,Generalised,Homo species
1,Q9NWB1,Protein_Pathway,R-HSA-9022707,Protein,NaN,Pathway,crossbar,Uniprot_ID,Reactome_ID,RNA binding protein fox-1 homolog 1,MECP2 regulates transcription factors,Generalised,Homo species


## 5. Standardise Fixed-Value Columns

In [10]:
final_df['head_type'] = 'Protein'
final_df['tail_type'] = 'Pathway'
final_df['relation']  = 'Protein_Pathway'

print("Unique relation:",  set(final_df['relation']))
print("Unique tail_id_is:", set(final_df['tail_id_is']))
print("Unique kg_source:", set(final_df['kg_source']))

Unique relation: {'Protein_Pathway'}
Unique tail_id_is: {'Reactome_ID', 'KEGG_ID'}
Unique kg_source: {'crossbar', 'TARKG'}


## 6. Deduplicate

In [11]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
})
print(f"After deduplication: {len(final_df_dedup):,} rows")

After deduplication: 65,444 rows


In [12]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,26562,0,26562
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,66,0,66


## 7. Fill Missing Protein Head Names from UniProt

In [13]:
# Normalise fake NaNs, fill head_detail_name from UniProt TrEMBL dict, drop remaining
final_df_dedup['head_detail_name'] = final_df_dedup['head_detail_name'].replace(['NAN', 'NaN', 'None'], np.nan)
final_df_dedup['head_detail_name'] = final_df_dedup['head_detail_name'].fillna(
    final_df_dedup['head'].map(uniprot_trembl_AC_Name_dict)
)

before = len(final_df_dedup)
final_df_dedup = final_df_dedup[~final_df_dedup['head_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df_dedup):,} rows with missing head_detail_name → {len(final_df_dedup):,} remaining")

Dropped 10 rows with missing head_detail_name → 65,434 remaining


## 8. Add Schema Columns and Enforce Column Order

In [14]:
final_df_dedup = final_df_dedup[REQUIRED_COLS]
print(f"Final shape: {final_df_dedup.shape}")
final_df_dedup.head()

Final shape: (65434, 13)


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,A0A075B6P5,Protein_Pathway,R-HSA-166663,Protein,None,Pathway,crossbar,Uniprot_ID,Reactome_ID,Immunoglobulin kappa variable 2-28 {ECO:000030...,Initial triggering of complement,Generalised,Homo species
1,A0A075B6P5,Protein_Pathway,R-HSA-173623,Protein,None,Pathway,crossbar,Uniprot_ID,Reactome_ID,Immunoglobulin kappa variable 2-28 {ECO:000030...,Classical antibody-mediated complement activation,Generalised,Homo species
2,A0A075B6P5,Protein_Pathway,R-HSA-202733,Protein,None,Pathway,crossbar,Uniprot_ID,Reactome_ID,Immunoglobulin kappa variable 2-28 {ECO:000030...,Cell surface interactions at the vascular wall,Generalised,Homo species
3,A0A075B6P5,Protein_Pathway,R-HSA-2029481,Protein,None,Pathway,crossbar,Uniprot_ID,Reactome_ID,Immunoglobulin kappa variable 2-28 {ECO:000030...,FCGR activation,Generalised,Homo species
4,A0A075B6P5,Protein_Pathway,R-HSA-2029485,Protein,None,Pathway,crossbar,Uniprot_ID,Reactome_ID,Immunoglobulin kappa variable 2-28 {ECO:000030...,Role of phospholipids in phagocytosis,Generalised,Homo species


## 9. QC — NaN Check

In [15]:
nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,26552,0,26552
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


## 10. Save Output

In [16]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_parquet(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 65,434 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/PROTEIN_PATHWAY/ALL_PROTEIN_PATHWAY.parquet


In [1]:
#